## Fetch old Match Details — 2023/24 & 2024/25

Loops through fixture files and fetches match details for every finished game.

**Before running:** update the `x-mas` token in the headers (grab a fresh one from DevTools on any FotMob match page).

In [ ]:
import json
import time
import requests
from pathlib import Path

# --- Config ---
FIXTURES_FILES = [
    # "fixtures_tiredness/eng_47_2023_2024_fixtures.json",
    "fixtures_tiredness/esp_87_2024_2025_fixtures.json",
]
OUTPUT_DIR = Path("raw_json/pl_match_details")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DELAY = 1.5

headers = {
    "authority": "www.fotmob.com",
    "method": "GET",
    "scheme": "https",
    "accept": "*/*",
    "accept-language": "en-US,en;q=0.9",
    "cache-control": "no-cache",
    "cookie": "_hjSessionUser_2585474=eyJpZCI6Ijg0MDFhMGVkLWI0MDItNWY0Yy1hMzkyLWM4YWQ3MGQyY2NhMiIsImNyZWF0ZWQiOjE3NjQ3MTk5ODQ2NTMsImV4aXN0aW5nIjp0cnVlfQ==; NEXT_LOCALE=en; _ga_G0V1WDW9B2=deleted; _gcl_au=1.1.327381733.1772502095; _ga=GA1.1.190417444.1764719984; _ga_SQ24F7Q7YW=GS2.1.s1773693825$o4$g0$t1773693827$j60$l0$h0; _ga_K2ECMCJBFQ=GS2.1.s1773693825$o4$g0$t1773693827$j59$l0$h0; _pubcid=a2ba22a3-3368-49c8-ae0c-37503b5afced; _cc_id=f1e79e0caf1a8a1a6d95cae482fd7bbc; cto_bundle=cufQNV9FWSUyQmRxSjk4aUF6cTBja0lxdDRHdHR3Z3ZnM1JqRk5nSWk0JTJGTTZGYVNCUiUyRndqJTJGRWQlMkJjcWhuVUElMkZtZkg5VlVNMGh3cCUyRkVpVHl2VXhaNXI1SzBPNDhqWTRvTXBXaEdHJTJCWkNjYVV3RG5lckxtczhDWE81NjlWSVlGZjdSbDNGejZYdGpIRXFkR0hzbHdHYnh3VXR4RnpsZVM4a2ppNnRVeEFNMWVtZmpQSkRXJTJCbWhwR25CejlleTBrZ1NIZjNKa1k; panoramaId_expiry=1780468754881; panoramaId=551bab106284817402eff1de7d764945a702ba48a61cb911d8515151174fa962; panoramaIdType=panoIndiv; u:location=%7B%22countryCode%22%3A%22US%22%2C%22regionId%22%3A%22CA%22%2C%22ip%22%3A%22127.0.0.1%22%2C%22ccode3%22%3A%22USA_CA%22%2C%22ccode3NoRegion%22%3A%22USA%22%2C%22timezone%22%3A%22America%2FLos_Angeles%22%7D; _hjSession_2585474=eyJpZCI6ImQ5OGI3YTk1LWYyNzktNGM2MS1iNGNhLTFiZDljOGZlM2NhMyIsImMiOjE3ODAwMDM4MzI4NDgsInMiOjAsInIiOjAsInNiIjowLCJzciI6MCwic2UiOjAsImZzIjowLCJzcCI6MX0=; turnstile_verified=1.1780003835.dad888c281ec1afab01d552e9ce52cb03212f65725c5263c8ada6cda8c788664; FCCDCF=%5Bnull%2Cnull%2Cnull%2Cnull%2Cnull%2Cnull%2C%5B%5B32%2C%22%5B%5C%22c3ecb5bf-1247-4068-8c8c-4346be5b0f29%5C%22%2C%5B1772315167%2C220000000%5D%5D%22%5D%5D%5D; FCNEC=%5B%5B%22AKsRol-gXWWEdUPDfsfDapuzTWXJ49hEqWcLtm629kefaiRxVejbFcFbj-MKQdW_ceJctR3zQpK-_Ef4bE7KYF-gZTA_EWq42PdH5uMxrdDF9MOs9AphgeHGaVe7DQDyglXYC1Q9sarttFiJi9XfkadyWw9ngEt9dA%3D%3D%22%5D%5D; g_state={\"i_l\":0,\"i_ll\":1780004627400,\"i_b\":\"MzOkSAL8Bn49XVIqRhipacC2XmSmALM3NHXo+OCRbCk\",\"i_e\":{\"enable_itp_optimization\":1},\"i_et\":1776627688235}; _ga_G0V1WDW9B2=GS2.1.s1780003832$o63$g1$t1780004631$j53$l0$h1363727128",
    "pragma": "no-cache",
    "priority": "u=1, i",
    "referer": "https://www.fotmob.com/matches/arsenal-vs-crystal-palace/36ytc8",
    "sec-ch-ua": '"Chromium";v="148", "Google Chrome";v="148", "Not/A)Brand";v="99"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"macOS"',
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "same-origin",
    "user-agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36",
    "x-mas": "eyJib2R5Ijp7InVybCI6Ii9hcGkvZGF0YS9tYXRjaERldGFpbHM/bWF0Y2hJZD00ODEzNzQ3IiwiY29kZSI6MTc4MDAwNDYzNDc1MCwiZm9vIjoicHJvZHVjdGlvbjoyMzdlMjI5OWM4MjI3ZDgxNWE5ZDNjZjM4Yjc3NmU1NmU5OGNiMWQwIn0sInNpZ25hdHVyZSI6IjUyQURBNDc3NDU5QzczMzc3NTQ0RUM3RDdBQkQ0QTJGIn0="
}

# --- Collect all finished match IDs across both seasons ---
all_matches = []
for fixtures_file in FIXTURES_FILES:
    with open(fixtures_file) as f:
        data = json.load(f)
    matches = data["data"]["fixtures"]["allMatches"]
    finished = [m for m in matches if m["status"].get("finished")]
    print(f"{fixtures_file}: {len(finished)} finished matches")
    all_matches.extend(finished)

print(f"\nTotal to fetch: {len(all_matches)}\n")

# --- Scrape ---
success = 0
failed = 0
skipped = 0

for i, match in enumerate(all_matches):
    match_id = match["id"]
    out_path = OUTPUT_DIR / f"{match_id}.json"

    # Skip if already fetched
    if out_path.exists():
        skipped += 1
        continue

    home = match["home"]["name"]
    away = match["away"]["name"]
    url = f"https://www.fotmob.com/api/data/matchDetails?matchId={match_id}"

    try:
        r = requests.get(url, headers=headers, timeout=10)

        if r.status_code != 200:
            print(f"[{i+1}/{len(all_matches)}] ✗ HTTP {r.status_code}: {match_id}")
            failed += 1
            continue

        data = r.json()

        league = data.get("general", {}).get("leagueName")
        finished_flag = data.get("general", {}).get("finished")

        if league != "Premier League" or not finished_flag:
            print(f"[{i+1}/{len(all_matches)}] ✗ Bad data (league='{league}', finished={finished_flag}): {match_id} — {home} vs {away}")
            failed += 1
            continue

        with open(out_path, "w") as f:
            json.dump({"pageProps": data}, f)

        print(f"[{i+1}/{len(all_matches)}] ✓ {home} vs {away} ({match_id})")
        success += 1

    except Exception as e:
        print(f"[{i+1}/{len(all_matches)}] ✗ Error on {match_id}: {e}")
        failed += 1

    time.sleep(DELAY)

print(f"\nDone. ✓ {success} saved | ✗ {failed} failed | ⏭ {skipped} skipped")


✓ Saved match details for 4506359
